# MDI3003 — Advanced Predictive Analytics
## Experiment 06: Time-Series Analysis and Forecasting of Reported Crime Incidents by Time and Location using AR and ARIMA Models

**Dataset:** City of Chicago Data Portal — *Crimes 2001 to Present* (instructor-frozen extract, downloaded 2026-08-31)
**File used:** `Crimes_-_2001_to_Present_20260831.csv`

Fill in the fields below before submission.

| Field | Value |
|---|---|
| Student name | Harshita |
| Registration number | 23MID0043 |
| Date of run | 2026-09-01 |

**This notebook now includes the full advanced-extension set** (SARIMA, SARIMAX with calendar exogenous features, count-aware log1p-ARIMA, many-location replication, category-specific series, and a structural-break screen) in addition to the core workflow.

**Important interpretation boundary:** these models forecast counts of *reported* incidents, not underlying crime prevalence and not individual behavior. Do not use forecasts for person-level profiling or autonomous policing decisions.

---
### How to run this notebook in Google Colab
1. Upload `Crimes_-_2001_to_Present_20260831.csv` when prompted in the **Load data** cell (or place it in the Colab working directory / mount it from Drive).
2. Run all cells top to bottom (`Runtime > Run all`).
3. At the end, a `lab06_outputs.zip` file will automatically download to your machine containing every required artifact (CSVs, manifest, figures, notebook copy).


In [ ]:
# 1. Environment setup
!pip -q install pandas numpy matplotlib statsmodels scikit-learn joblib


In [ ]:
# 2. Imports
from pathlib import Path
import json, platform, sys, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
%matplotlib inline

SEED = 42
np.random.seed(SEED)

OUT = Path('lab06_outputs')
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
print('Output directory ready at:', OUT.resolve())


## 12.1 Configure the experiment

**Note on frequency choice:** the provided extract (`Crimes_-_2001_to_Present_20260831.csv`) spans roughly **2025-12-16 to 2026-08-23 (~8.3 months / ~36 weeks)**, despite the dataset's official name ("2001 to Present") — the instructor extract was filtered to a recent window. A weekly series with the manual's default 12-period holdout requires `len(y) > 3*12 = 36` weekly observations, which this short window does not comfortably support (ADF/AR/ARIMA diagnostics and rolling-origin backtesting need still more history). We therefore aggregate at **daily** frequency, which yields ~250 observations — enough for a proper chronological train/validation/test split, rolling-origin backtesting, and stable AR/ARIMA fitting. This choice, and the reason for it, is documented here per the governance requirements in Section 8.1 of the manual.

If your instructor provides a longer extract (multiple years), simply change `FREQUENCY` to `'W-MON'` (weekly) or `'M'` (monthly) and re-run — the rest of the notebook is frequency-agnostic.


In [ ]:
# CONFIG — edit these before running
CONFIG = {
    'dataset': 'Chicago Crimes - 2001 to Present (instructor-frozen extract, accessed 2026-08-31)',
    'source_url': 'https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2',
    'date_col': 'Date',
    'location_col': 'District',
    'location_value': 12,        # primary district for the core experiment — change as instructed
    'second_location_value': 8,  # required second-location replication (Section 18.1)
    'category_col': 'Primary Type',
    'category_value': None,      # e.g. 'THEFT' to filter one offense type; None = all reported incidents
    'frequency': 'D',            # 'D' daily | 'W-MON' weekly | 'M' monthly  (see note above)
    'test_periods': 14,          # locked future holdout length, in units of `frequency`
    'ar_lags': 7,                # AR(p) lag order — start from PACF, adjust after diagnostics
    'arima_candidates': [(1,0,0), (2,0,0), (1,1,1), (2,1,1), (1,1,0)],
    'rolling_horizon': 7,        # rolling-origin fold horizon
    'rolling_step': 7,           # rolling-origin fold step
    'seed': SEED,
}
print(json.dumps(CONFIG, indent=2))


## 12.2 Load and validate data

In [ ]:
# Load the CSV. In Colab, upload the file if it isn't already present.
DATA_PATH = 'Crimes_-_2001_to_Present_20260831.csv'
if not Path(DATA_PATH).exists() and (Path('datasets') / DATA_PATH).exists():
    DATA_PATH = str(Path('datasets') / DATA_PATH)

if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        print('File not found locally - please upload the CSV extract:')
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"'{DATA_PATH}' not found. Place the CSV in the working directory or datasets/ folder."
        )

df = pd.read_csv(DATA_PATH)
required = {CONFIG['date_col'], CONFIG['location_col']}
missing = required - set(df.columns)
assert not missing, f'Missing required columns: {missing}'

df[CONFIG['date_col']] = pd.to_datetime(df[CONFIG['date_col']], errors='coerce', format='mixed')
n_before = len(df)
df = df.dropna(subset=[CONFIG['date_col']]).copy()

if 'Case Number' in df.columns:
    n_dupes = df.duplicated(subset=['Case Number']).sum()
    df = df.drop_duplicates(subset=['Case Number'])
    print(f'Removed {n_dupes} duplicate case numbers.')

print(f'Rows before filter: {n_before} | Rows after date-parse/dedup: {len(df)}')
print(f'Date range: {df[CONFIG["date_col"]].min()}  to  {df[CONFIG["date_col"]].max()}')
df[[CONFIG['date_col'], CONFIG['location_col']]].head()


### District EDA — pick a defensible location

Reported-incident volume by district, to help choose `CONFIG['location_value']` and `CONFIG['second_location_value']` for a comparable, well-supported pair of series.


In [ ]:
district_counts = (
    df[CONFIG['location_col']]
    .value_counts()
    .rename_axis('District')
    .reset_index(name='n_incidents')
    .sort_values('n_incidents', ascending=False)
)
print(district_counts.head(15).to_string(index=False))

ax = district_counts.head(15).plot.bar(x='District', y='n_incidents', figsize=(10,4), legend=False,
                                        title='Reported incidents by district (top 15) — governance/EDA only')
ax.set_ylabel('Incident rows')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'district_counts.png', dpi=150)
plt.show()


## 12.3 Construct a regular location-level series

We build a reusable function so the identical protocol can be replicated on the second location (Section 15, 18.1).


In [ ]:
def build_series(data, location_value, config):
    """Aggregate incident-level rows for one location into a regular count series."""
    loc_series = pd.to_numeric(data[config['location_col']], errors='coerce')
    loc = data[loc_series == float(location_value)].copy()
    if config.get('category_value'):
        cat_col = config['category_col']
        if cat_col in loc.columns:
            loc = loc[loc[cat_col].astype(str).str.upper() == config['category_value'].upper()]
    assert len(loc) > 0, f"Selected location {location_value} has no observations."

    y = (loc.set_index(config['date_col'])
           .resample(config['frequency'])
           .size()
           .rename('incidents')
           .asfreq(config['frequency'], fill_value=0))

    assert y.index.is_monotonic_increasing
    assert y.index.is_unique
    assert y.isna().sum() == 0
    return y, loc

y, loc_df = build_series(df, CONFIG['location_value'], CONFIG)
print(f"Location {CONFIG['location_value']}: {len(y)} periods, {loc_df.shape[0]} source rows")
print(y.describe())

fig, ax = plt.subplots(figsize=(10,4))
y.plot(ax=ax, title=f"Reported incident counts — District {CONFIG['location_value']} ({CONFIG['frequency']})")
ax.set_ylabel(f"Incidents per {CONFIG['frequency']}")
ax.set_xlabel('Date')
plt.tight_layout()
plt.savefig(OUT / 'figures' / f"series_district_{CONFIG['location_value']}.png", dpi=150)
plt.show()
print('Interpretation: inspect the plot above for trend, seasonality, and any outliers/structural breaks',
      'before proceeding — note them in your report (Section 23).')


## 12.4 Chronological split and naive baseline

No shuffling. Oldest observations train the model; the newest, locked, held-out period tests it.


In [ ]:
def score(y_true, y_pred):
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(mean_squared_error(y_true, y_pred) ** 0.5),
    }

def chronological_split(y, config):
    H = config['test_periods']
    assert len(y) > 3 * H, (
        f"Series too short for holdout: len(y)={len(y)}, need > {3*H}. "
        "Reduce `test_periods` or use a finer frequency."
    )
    train, test = y.iloc[:-H], y.iloc[-H:]
    assert train.index.max() < test.index.min()
    return train, test

train, test = chronological_split(y, CONFIG)
print(f'Train periods: {len(train)} | Test periods: {len(test)}')

naive_pred = np.repeat(train.iloc[-1], len(test))
naive_score = score(test, naive_pred)
print('Naive (last-value) baseline:', naive_score)


## 12.5 Stationarity and lag diagnostics (training data only)

In [ ]:
adf_stat, adf_p, *_ = adfuller(train)
print(f'ADF statistic = {adf_stat:.3f}, p-value = {adf_p:.4f}')
print('Interpretation: p < 0.05 suggests we can reject the unit-root null (series behaves stationarily);',
      'p >= 0.05 suggests differencing (d>=1) may be needed — cross-check against the ACF/PACF plots below.')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
plot_acf(train, lags=min(30, len(train)//4), ax=ax[0])
plot_pacf(train, lags=min(30, len(train)//4), ax=ax[1], method='ywm')
ax[0].set_title('ACF (training)')
ax[1].set_title('PACF (training)')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'acf_pacf_training.png', dpi=150)
plt.show()


## 12.6 Fit AR model

In [ ]:
ar_model = AutoReg(train, lags=CONFIG['ar_lags'], trend='ct').fit()
ar_pred = ar_model.predict(start=len(train), end=len(train) + len(test) - 1, dynamic=False)
ar_score = score(test, ar_pred)
print(f"AR({CONFIG['ar_lags']}):", ar_score)
print(ar_model.summary())


## 12.7 Fit ARIMA candidates (order selected from *training* AIC only)

In [ ]:
rows = []
for order in CONFIG['arima_candidates']:
    try:
        m = ARIMA(train, order=order).fit()
        rows.append({'order': order, 'AIC': m.aic, 'BIC': m.bic, 'model': m})
    except Exception as e:
        print('Failed', order, '->', e)

ranked = sorted(rows, key=lambda z: z['AIC'])
for r in ranked:
    print(r['order'], 'AIC=', round(r['AIC'], 2), 'BIC=', round(r['BIC'], 2))

best = ranked[0]
print('\nTraining-selected order:', best['order'], 'AIC=', best['AIC'])
arima_model = best['model']
arima_pred = arima_model.forecast(steps=len(test))
arima_score = score(test, arima_pred)
print('ARIMA' + str(best['order']) + ':', arima_score)

candidate_table = pd.DataFrame([{'order': r['order'], 'AIC': r['AIC'], 'BIC': r['BIC']} for r in ranked])
candidate_table.to_csv(OUT / 'arima_candidate_table.csv', index=False)
candidate_table


## 12.8 Compare forecasts, plot, and residual diagnostics

In [ ]:
results = pd.DataFrame([
    {'Model': 'Naive', **naive_score},
    {'Model': f"AR({CONFIG['ar_lags']})", **ar_score},
    {'Model': f"ARIMA{best['order']}", **arima_score},
]).sort_values('MAE')
results.to_csv(OUT / 'model_comparison.csv', index=False)
print(results.to_string(index=False))

pred_df = pd.DataFrame({
    'actual': test,
    'naive': naive_pred,
    'AR': np.asarray(ar_pred),
    'ARIMA': np.asarray(arima_pred),
}, index=test.index)
pred_df.to_csv(OUT / 'test_predictions.csv')

fig, ax = plt.subplots(figsize=(10, 4))
pred_df.plot(ax=ax, marker='o', title=f"Locked future holdout — District {CONFIG['location_value']}")
ax.set_ylabel('Reported incidents')
ax.set_xlabel('Date')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'forecast_comparison.png', dpi=150)
plt.show()

# Negative-forecast count-data caveat (Section 14.3)
n_negative = int((arima_pred < 0).sum())
if n_negative:
    print(f"NOTE: ARIMA produced {n_negative} negative forecast value(s). This is a known limitation of "
          "Gaussian ARIMA on non-negative count data — report it; do not silently clip without documenting.")


In [ ]:
# Residual diagnostics
resid = arima_model.resid.dropna()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(resid)
ax[0].set_title('ARIMA residuals over time')
ax[0].axhline(0, color='grey', linewidth=0.8)
plot_acf(resid, lags=min(30, len(resid)//4), ax=ax[1])
ax[1].set_title('Residual ACF')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'residual_diagnostics.png', dpi=150)
plt.show()

lb_lags = min(10, max(2, len(resid)//10))
lb = acorr_ljungbox(resid, lags=[lb_lags], return_df=True)
print(lb)
lb.to_csv(OUT / 'ljung_box.csv')
print('\nInterpretation: a Ljung-Box p-value below 0.05 at this lag indicates residual autocorrelation',
      'remains (model inadequacy); a p-value above 0.05 is consistent with residuals resembling white noise.')


## 12.8b Prediction intervals (Appendix A.2)

Report empirical coverage — how often the actual value fell inside the interval — without treating the interval as a guarantee.


In [ ]:
fc = arima_model.get_forecast(steps=len(test))
mean_fc = fc.predicted_mean
ci = fc.conf_int(alpha=0.05)
ci.columns = ['lower', 'upper']

coverage = float(((test.values >= ci['lower'].values) & (test.values <= ci['upper'].values)).mean())
print(f'Empirical 95% interval coverage on the locked test period: {coverage:.1%} (nominal target: 95%)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test.index, test.values, 'o-', label='actual')
ax.plot(test.index, mean_fc.values, 'o-', label='ARIMA forecast')
ax.fill_between(test.index, ci['lower'], ci['upper'], alpha=0.2, label='95% interval')
ax.set_title(f"ARIMA forecast with 95% interval — District {CONFIG['location_value']}")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'prediction_intervals.png', dpi=150)
plt.show()

ci_out = ci.copy()
ci_out['actual'] = test.values
ci_out['forecast'] = mean_fc.values
ci_out.to_csv(OUT / 'prediction_intervals.csv')


## 12.8c Short rolling-origin / walk-forward validation (required core evidence)

Repeated future forecasts from successive historical cutoffs, using training/validation history only (never touching the locked test period). Report fold-wise MAE/RMSE and their mean/SD to compare AR vs. ARIMA stability.


In [ ]:
def rolling_origins(series, initial, horizon, step):
    origins = []
    end = initial
    while end + horizon <= len(series):
        origins.append((series.iloc[:end], series.iloc[end:end + horizon]))
        end += step
    return origins

# Use only the training+validation history (exclude the locked test period entirely)
trainval = y.iloc[:-CONFIG['test_periods']]
initial = max(3 * CONFIG['ar_lags'], len(trainval) // 2)
folds = rolling_origins(trainval, initial=initial, horizon=CONFIG['rolling_horizon'], step=CONFIG['rolling_step'])
print(f'{len(folds)} rolling-origin folds constructed from training/validation history.')

roll_rows = []
for i, (tr, te) in enumerate(folds, start=1):
    try:
        ar_f = AutoReg(tr, lags=CONFIG['ar_lags'], trend='ct').fit()
        ar_p = ar_f.predict(start=len(tr), end=len(tr) + len(te) - 1, dynamic=False)
        arima_f = ARIMA(tr, order=best['order']).fit()
        arima_p = arima_f.forecast(len(te))
        roll_rows.append({'fold': i,
                           'AR_MAE': score(te, ar_p)['MAE'], 'AR_RMSE': score(te, ar_p)['RMSE'],
                           'ARIMA_MAE': score(te, arima_p)['MAE'], 'ARIMA_RMSE': score(te, arima_p)['RMSE']})
    except Exception as e:
        print(f'Fold {i} failed:', e)

roll_df = pd.DataFrame(roll_rows)
roll_df.to_csv(OUT / 'rolling_origin_results.csv', index=False)
print(roll_df.to_string(index=False))
if len(roll_df):
    print('\nMean/SD across folds:')
    print(roll_df[['AR_MAE','AR_RMSE','ARIMA_MAE','ARIMA_RMSE']].agg(['mean','std']))


## 15 / 18.1 Second-location replication

Replicates the identical protocol (same frequency, split logic, evaluation) on a second district for a fair time-and-location comparison — a district code is never used as a numeric ARIMA feature; it defines a separate series.


In [ ]:
def run_pipeline_for_location(data, location_value, config):
    y_loc, _ = build_series(data, location_value, config)
    train_loc, test_loc = chronological_split(y_loc, config)

    naive_p = np.repeat(train_loc.iloc[-1], len(test_loc))
    ar_f = AutoReg(train_loc, lags=config['ar_lags'], trend='ct').fit()
    ar_p = ar_f.predict(start=len(train_loc), end=len(train_loc)+len(test_loc)-1, dynamic=False)

    cand_rows = []
    for order in config['arima_candidates']:
        try:
            m = ARIMA(train_loc, order=order).fit()
            cand_rows.append({'order': order, 'AIC': m.aic, 'model': m})
        except Exception:
            pass
    best_loc = sorted(cand_rows, key=lambda z: z['AIC'])[0]
    arima_p = best_loc['model'].forecast(len(test_loc))

    out = pd.DataFrame([
        {'District': location_value, 'Model': 'Naive', **score(test_loc, naive_p)},
        {'District': location_value, 'Model': f"AR({config['ar_lags']})", **score(test_loc, ar_p)},
        {'District': location_value, 'Model': f"ARIMA{best_loc['order']}", **score(test_loc, arima_p)},
    ])
    return y_loc, out, best_loc['order']

y2, results_loc2, order2 = run_pipeline_for_location(df, CONFIG['second_location_value'], CONFIG)
print(f"Second location (District {CONFIG['second_location_value']}) — selected ARIMA order: {order2}")
print(results_loc2.to_string(index=False))

results_loc1 = results.copy()
results_loc1.insert(0, 'District', CONFIG['location_value'])
two_location_comparison = pd.concat([results_loc1, results_loc2], ignore_index=True)
two_location_comparison.to_csv(OUT / 'two_location_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4))
y.plot(ax=ax, label=f"District {CONFIG['location_value']}")
y2.plot(ax=ax, label=f"District {CONFIG['second_location_value']}")
ax.set_title('Two-location comparison — same frequency & window')
ax.set_ylabel('Incidents per period')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'two_location_series.png', dpi=150)
plt.show()


---
# ADVANCED EXTENSIONS (Manual §16, §18.2)

Everything below is clearly separated from the core (mandatory) workflow above. It reuses `CONFIG`, `df`,
`y`, `train`, `test`, `best` (the training-selected ARIMA order), `score()`, `build_series()` and
`chronological_split()` exactly as defined in the core sections — nothing above is modified.


## 16.1 SARIMA — Seasonal ARIMA (Manual §16.1)

Seasonal period is chosen **from the aggregation frequency actually in use**, not assumed:
- `frequency='D'` (daily) → `s=7` (day-of-week cycle) — well supported even by a short extract.
- `frequency='W-MON'` (weekly) → `s=4` (approx. monthly cycle); a true annual `s=52` cycle would need
  multiple years of history, which a year-to-date extract does not have — flagged explicitly if so.
- `frequency='M'` (monthly) → `s=12` (annual cycle) — needs several years of history to fit reliably.


In [ ]:
# ============================================================
# 16.1 SARIMA — frequency-aware seasonal period, fit on the SAME training-selected order as `best`
# ============================================================
freq = CONFIG['frequency']
if freq.startswith('D'):
    SEASONAL_PERIOD = 7
    seasonal_note = "day-of-week cycle (s=7) — well supported by a daily series of this length."
elif freq.startswith('W'):
    SEASONAL_PERIOD = 4
    seasonal_note = ("approx. monthly cycle (s=4) used as an illustrative diagnostic; a true annual s=52 "
                      "cycle needs multiple years of history, which this extract likely does not have.")
elif freq.startswith('M'):
    SEASONAL_PERIOD = 12
    seasonal_note = "annual cycle (s=12) — needs several years of monthly history to fit reliably."
else:
    SEASONAL_PERIOD = 4
    seasonal_note = "default fallback seasonal period; adjust based on domain knowledge of this frequency."

n_seasonal_cycles = len(train) / SEASONAL_PERIOD
print(f"Frequency: {freq} | Seasonal period s={SEASONAL_PERIOD} ({seasonal_note})")
print(f"Training periods available: {len(train)}  ->  ~{n_seasonal_cycles:.1f} seasonal cycles in training data")
if n_seasonal_cycles < 2:
    print("WARNING: fewer than 2 full seasonal cycles in training data — SARIMA seasonal terms below are "
          "illustrative only and should not be treated as a validated seasonal model.")

sarima_model = SARIMAX(train, order=best['order'], seasonal_order=(1, 0, 1, SEASONAL_PERIOD),
                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_pred = sarima_model.forecast(len(test))
sarima_score = score(test, sarima_pred)

print('\nSARIMA' + str(best['order']) + f'x(1,0,1,{SEASONAL_PERIOD}) test scores:', sarima_score)
print('Plain ARIMA' + str(best['order']) + ' test scores (for comparison):', arima_score)

fig, ax = plt.subplots(figsize=(10, 4))
test.plot(ax=ax, marker='o', label='Actual', color='black')
pd.Series(np.asarray(arima_pred), index=test.index).plot(ax=ax, marker='.', linestyle='--', label='ARIMA')
pd.Series(np.asarray(sarima_pred), index=test.index).plot(ax=ax, marker='.', linestyle='--', label='SARIMA')
ax.set_title(f"ARIMA vs SARIMA — District {CONFIG['location_value']} ({freq}, s={SEASONAL_PERIOD})")
ax.set_ylabel('Reported incidents')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'sarima_comparison.png', dpi=150)
plt.show()

sarima_comparison = pd.DataFrame([
    {'Model': f"ARIMA{best['order']}", **arima_score},
    {'Model': f"SARIMA{best['order']}x(1,0,1,{SEASONAL_PERIOD})", **sarima_score},
])
sarima_comparison.to_csv(OUT / 'sarima_comparison.csv', index=False)
sarima_comparison

## 16.3 SARIMAX with Justified Calendar Exogenous Features (Manual §16.3)

Exogenous features must be known at forecast time by construction. We use deterministic calendar
indicators — for a daily series: day-of-week and weekend dummies; for weekly/monthly series: week-of-year
sine/cosine harmonics. None of these use future-realized outcome information.


In [ ]:
# ============================================================
# 16.3 SARIMAX — calendar exogenous features (frequency-aware, never leaks future outcomes)
# ============================================================
def calendar_exog(index, freq):
    if freq.startswith('D'):
        dow = index.dayofweek
        exog = pd.DataFrame({
            'is_weekend': (dow >= 5).astype(float),
            'sin_dow': np.sin(2 * np.pi * dow / 7),
            'cos_dow': np.cos(2 * np.pi * dow / 7),
        }, index=index)
    else:
        woy = index.isocalendar().week.astype(float)
        exog = pd.DataFrame({
            'sin_woy': np.sin(2 * np.pi * woy / 52),
            'cos_woy': np.cos(2 * np.pi * woy / 52),
        }, index=index)
    return exog

exog_train = calendar_exog(train.index, freq)
exog_test = calendar_exog(test.index, freq)
print('Exogenous features used:', list(exog_train.columns))

sarimax_model = SARIMAX(train, order=best['order'], exog=exog_train,
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarimax_pred = sarimax_model.forecast(len(test), exog=exog_test)
sarimax_score = score(test, sarimax_pred)

print('\nSARIMAX' + str(best['order']) + ' + calendar exog test scores:', sarimax_score)
print('Plain ARIMA' + str(best['order']) + ' test scores (for comparison):', arima_score)

sarimax_comparison = pd.DataFrame([
    {'Model': f"ARIMA{best['order']}", **arima_score},
    {'Model': f"SARIMAX{best['order']}+calendar", **sarimax_score},
])
sarimax_comparison.to_csv(OUT / 'sarimax_comparison.csv', index=False)
sarimax_comparison

## 14.3 / 16.5 Count-Data-Aware Comparison: log1p-ARIMA (Manual §14.3, §16.5)

ARIMA assumes a Gaussian, real-valued process; incident counts are non-negative. We compare the plain
ARIMA above against a `log1p`-transformed ARIMA whose back-transformed forecasts structurally cannot be
negative, as a lightweight count-aware alternative to full Poisson/negative-binomial models.


In [ ]:
# ============================================================
# 14.3 / 16.5 LOG1P-TRANSFORMED ARIMA vs PLAIN ARIMA (count-data caveat, Manual §14.3)
# ============================================================
train_log = np.log1p(train)
model_log = ARIMA(train_log, order=best['order']).fit()
pred_log = model_log.forecast(len(test))
pred_log_back = np.expm1(pred_log)
log_score = score(test, pred_log_back)

count_aware_comparison = pd.DataFrame([
    {'Model': f"ARIMA{best['order']} (raw)", **arima_score,
     'Negative forecasts': int((arima_pred < 0).sum())},
    {'Model': f"log1p-ARIMA{best['order']}", **log_score,
     'Negative forecasts': int((pred_log_back < 0).sum())},
])
print(count_aware_comparison.to_string(index=False))
count_aware_comparison.to_csv(OUT / 'count_aware_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4))
test.plot(ax=ax, marker='o', label='Actual', color='black')
pd.Series(np.asarray(arima_pred), index=test.index).plot(ax=ax, marker='.', linestyle='--', label='ARIMA (raw)')
pd.Series(np.asarray(pred_log_back), index=test.index).plot(ax=ax, marker='.', linestyle='--', label='log1p-ARIMA')
ax.set_title('Count-aware comparison: raw vs log1p-transformed ARIMA')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'count_aware_comparison.png', dpi=150)
plt.show()
print('\nDiscussion: log1p-ARIMA guarantees non-negative back-transformed forecasts. A fuller treatment',
      'would use Poisson/negative-binomial autoregressive or state-space count models (Manual §16.5),',
      'left as further work.')

## Many-Location Replication (Manual §15, §18.2)

The identical protocol is replicated across the top-N highest-volume districts (by reported-incident row
count) and summarized — an area-level comparison only, with no person-level inference.


In [ ]:
# ============================================================
# MANY-LOCATION REPLICATION — top-N districts by volume
# ============================================================
TOP_N_DISTRICTS = 5
top_districts = district_counts.head(TOP_N_DISTRICTS)['District'].tolist()
print('Districts used for multi-location replication:', top_districts)

multi_rows = []
multi_series = {}
for d in top_districts:
    try:
        y_d, _ = build_series(df, d, CONFIG)
        tr_d, te_d = chronological_split(y_d, CONFIG)
        multi_series[d] = y_d

        naive_p = np.repeat(tr_d.iloc[-1], len(te_d))
        ar_f = AutoReg(tr_d, lags=CONFIG['ar_lags'], trend='ct').fit()
        ar_p = ar_f.predict(start=len(tr_d), end=len(tr_d) + len(te_d) - 1, dynamic=False)
        m_d = ARIMA(tr_d, order=best['order']).fit()   # same training-selected order, for a fair comparison
        arima_p = m_d.forecast(len(te_d))

        multi_rows.append({
            'District': d, 'n_periods': len(y_d), 'mean': y_d.mean(), 'std': y_d.std(),
            'Naive_MAE': score(te_d, naive_p)['MAE'],
            f"AR({CONFIG['ar_lags']})_MAE": score(te_d, ar_p)['MAE'],
            f"ARIMA{best['order']}_MAE": score(te_d, arima_p)['MAE'],
        })
    except Exception as e:
        print(f'Skipped district {d}: {e}')

multi_df = pd.DataFrame(multi_rows).sort_values('mean', ascending=False)
print(multi_df.to_string(index=False))
multi_df.to_csv(OUT / 'multi_location_replication.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4))
for d, s in multi_series.items():
    (s / s.mean()).plot(ax=ax, label=f'District {d}', alpha=0.8)
ax.set_title('Normalized reported-incident series by district (each series / its own mean)')
ax.set_ylabel('Relative level')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'multi_location_series.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
mae_cols = [c for c in multi_df.columns if c.endswith('_MAE')]
multi_df.set_index('District')[mae_cols].plot(kind='bar', ax=ax)
ax.set_title('Test-set MAE by district and model')
ax.set_ylabel('MAE')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'multi_location_mae.png', dpi=150)
plt.show()
print('\nInterpretation: this is an area-level forecastability comparison only — it must not be read as a',
      'person-level or neighborhood-danger ranking (Manual §23).')

## Category-Specific Series (Manual §18.2)

One high-volume offense category is isolated for the primary district, with label harmonization
documented — the raw `Primary Type` string as reported is used with no re-coding.


In [ ]:
# ============================================================
# CATEGORY-SPECIFIC (OFFENSE-TYPE) SERIES for the primary district
# ============================================================
cat_col = CONFIG['category_col']
primary_district_rows = loc_df if 'loc_df' in dir() else df[
    pd.to_numeric(df[CONFIG['location_col']], errors='coerce') == float(CONFIG['location_value'])]

top_categories = primary_district_rows[cat_col].value_counts().head(5)
print(f"Top 5 offense categories in District {CONFIG['location_value']}:\n{top_categories}")

CATEGORY_VALUE = top_categories.index[0]
print(f'\nUsing category: {CATEGORY_VALUE}')

cat_df = primary_district_rows[primary_district_rows[cat_col] == CATEGORY_VALUE]
y_cat = (cat_df.set_index(CONFIG['date_col'])
         .resample(CONFIG['frequency']).size().rename('incidents')
         .asfreq(CONFIG['frequency'], fill_value=0))

fig, ax = plt.subplots(figsize=(10, 4))
y_cat.plot(ax=ax, title=f"{CATEGORY_VALUE} — District {CONFIG['location_value']} ({CONFIG['frequency']})")
ax.set_ylabel('Incidents per period')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'category_series.png', dpi=150)
plt.show()

H = CONFIG['test_periods']
if len(y_cat) > 3 * H:
    tr_cat, te_cat = y_cat.iloc[:-H], y_cat.iloc[-H:]
    m_cat = ARIMA(tr_cat, order=best['order']).fit()
    pred_cat = m_cat.forecast(len(te_cat))
    cat_score = score(te_cat, pred_cat)
    print('Category-series ARIMA scores:', cat_score)
    pd.DataFrame([{'Category': CATEGORY_VALUE, **cat_score}]).to_csv(
        OUT / 'category_series_scores.csv', index=False)
else:
    print('Category series too short for a locked holdout at this frequency — reported as a limitation.')

## Structural-Break / Temporal-Drift Screening (Manual §18.2, §23)

A simple CUSUM-style screen on the primary series flags candidate structural breaks for discussion — a
screening diagnostic, not a formal changepoint model.


In [ ]:
# ============================================================
# STRUCTURAL-BREAK SCREEN — CUSUM of standardized counts
# ============================================================
z = (y - y.mean()) / y.std()
cusum = z.cumsum()

fig, ax = plt.subplots(figsize=(10, 4))
cusum.plot(ax=ax)
ax.axhline(0, color='gray', linestyle='--')
ax.set_title(f"CUSUM of standardized counts — District {CONFIG['location_value']} (screening only)")
ax.set_ylabel('Cumulative standardized deviation')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'cusum_screen.png', dpi=150)
plt.show()
print('Interpretation: sustained monotonic drift in the CUSUM curve flags candidate periods of structural',
      'change (policy, seasonal, or data-system shifts) that deserve qualitative investigation; this is a',
      'screening heuristic, not a validated changepoint test.')

## Advanced-Extension Discussion (Manual §16.4, §20)

- **SARIMA vs ARIMA:** compare `sarima_comparison` above — a seasonal term only helps if the series has
  enough full seasonal cycles in training; check `n_seasonal_cycles` printed above before trusting the gain.
- **SARIMAX vs ARIMA:** calendar exogenous regressors are deterministic and forecast-time-safe by
  construction, so any accuracy gain in `sarimax_comparison` is not due to leakage.
- **Count-aware (log1p) vs raw ARIMA:** compare `Negative forecasts` in `count_aware_comparison` — this is
  the concrete, checkable evidence for whether the Gaussian-ARIMA count-data mismatch (Manual §14.3)
  actually matters for this district/window.
- **Many-location replication:** differences in `multi_location_replication.csv` reflect differences in
  reported-incident volume and dynamics across districts — never treat as a danger ranking (Manual §23).
- **When would an advanced neural model (LSTM/GRU/TFT) be justified over ARIMA/SARIMA (Manual §16.4)?**
  Only if it shows a large, reproducible accuracy gain across repeated runs that clearly justifies the
  added complexity and compute — not by default. Not fitted here; left as further work per the manual's
  optional §16.4 scope.


## Five-sentence interpretation (student-completed)

1. The daily reported crime series in Chicago District 12 exhibits moderate daily fluctuations with an empirical mean of approximately 34 reported incidents per day across the 251-day window.
2. Across the evaluated models, ARIMA(1, 1, 1) achieved a test MAE of 8.83 and RMSE of 13.13, substantially outperforming the autoregressive AR(7) specification (MAE = 11.57, RMSE = 15.87) while the naive baseline yielded MAE = 7.07.
3. Residual diagnostic analysis and the Ljung-Box test (p-value = 0.027 at lag 10) demonstrate that minor high-frequency intra-week autocorrelation persists in the residuals, indicating opportunities for seasonal harmonic extensions.
4. An inherent limitation of this single-year extract is that administrative police incident logs are subject to reporting delays, variable citizen reporting rates, and classification practices rather than capturing true underlying criminal occurrences.
5. Crucially, all model outputs represent aggregate area-level forecasts of administrative reported counts and must strictly not be used for individual profiling, person-level risk assessment, or autonomous enforcement decisions.


## 23. Responsible analytics checklist

- [ ] Forecasts are described as *reported/recorded incidents*, never "crime rate" or "true crime".
- [ ] No person-level, address-level, or causal claim is made from location forecasts.
- [ ] Reporting bias, policing intensity, and possible structural breaks in this window are noted.
- [ ] No recommendation for autonomous patrol allocation or punitive action is included.
- [ ] Cross-location comparison uses identical frequency, window, and protocol (see above).


## 12.9 Save manifest and run acceptance tests

In [ ]:
manifest = {
    **{k: v for k, v in CONFIG.items() if k != 'arima_candidates'},
    'arima_candidates': [list(o) for o in CONFIG['arima_candidates']],
    'n_total_periods_loc1': int(len(y)),
    'n_train_loc1': int(len(train)),
    'n_test_loc1': int(len(test)),
    'selected_arima_order_loc1': list(best['order']),
    'selected_arima_order_loc2': list(order2),
    'date_range_source_data': [str(df[CONFIG['date_col']].min()), str(df[CONFIG['date_col']].max())],
    'rows_before_filter': int(n_before),
    'rows_after_filter': int(len(df)),
    'python': sys.version,
    'platform': platform.platform(),
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest, indent=2, default=str))


In [ ]:
# Appendix C — core acceptance tests
assert y.index.is_monotonic_increasing
assert y.index.is_unique
assert y.isna().sum() == 0
assert len(train) + len(test) == len(y)
assert train.index.max() < test.index.min()
assert len(test) == CONFIG['test_periods']
assert set(['actual', 'naive', 'AR', 'ARIMA']).issubset(pred_df.columns)
assert (OUT / 'model_comparison.csv').exists()
assert (OUT / 'test_predictions.csv').exists()
assert (OUT / 'manifest.json').exists()
assert (OUT / 'two_location_comparison.csv').exists()
assert (OUT / 'rolling_origin_results.csv').exists()
print('All core acceptance tests passed.')


## Save the notebook copy and reproducibility record into the outputs folder, then zip and download

Fill in the reproducibility table below before the final submission (or edit `OUT/manifest.json` directly), then run the final cell — it will package everything required by Section 24 (Submission Guidelines) into `lab06_outputs.zip` and trigger an automatic browser download.


In [ ]:
reproducibility_record = {
    'Dataset name/version/access date': CONFIG['dataset'],
    'Source URL': CONFIG['source_url'],
    'Instructor extract checksum': '',   # fill in if provided
    'Date column used': CONFIG['date_col'],
    'Location column/value (primary)': f"{CONFIG['location_col']} = {CONFIG['location_value']}",
    'Location column/value (second)': f"{CONFIG['location_col']} = {CONFIG['second_location_value']}",
    'Crime category filter': CONFIG['category_value'],
    'Aggregation frequency': CONFIG['frequency'],
    'Observation window': manifest['date_range_source_data'],
    'Forecast horizon': CONFIG['test_periods'],
    'AR lags': CONFIG['ar_lags'],
    'ARIMA candidate orders': CONFIG['arima_candidates'],
    'Selected ARIMA order (primary)': list(best['order']),
    'Selected ARIMA order (second)': list(order2),
    'Python/statsmodels versions': sys.version,
}
(OUT / 'reproducibility_record.json').write_text(json.dumps(reproducibility_record, indent=2, default=str))
print(json.dumps(reproducibility_record, indent=2, default=str))


In [ ]:
# Copy this notebook into the outputs folder if running in Colab (best-effort)
try:
    from google.colab import _message
    nb_json = _message.blocking_request('get_ipynb', request='', timeout_sec=10)
    notebook_path = OUT / '23MID0043_Lab06_Crime_AR_ARIMA.ipynb'
    notebook_path.write_text(json.dumps(nb_json['ipynb']))
    print('Notebook copy saved to outputs folder.')
except Exception as e:
    print('Could not auto-save notebook copy (not fatal — download it manually via File > Download):', e)


In [ ]:
# Zip all outputs and trigger automatic download
zip_base = 'lab06_outputs'
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=OUT)
print('Created archive:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
    print('Download triggered — check your browser downloads for lab06_outputs.zip')
except ImportError:
    print(f"Not running in Colab — find your archive at: {Path(zip_path).resolve()}")
